# Backtesting Pairs Trading Strategy

This notebook evaluates the performance of the statistical arbitrage strategy using the selected cointegrated pair.

Steps performed:
1. Load cleaned price data
2. Construct spread between two stocks
3. Generate trading signals using Z-score
4. Compute strategy returns
5. Evaluate performance metrics


## Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Cleaned Price Data


In [ ]:
# Load processed price dataset
close_prices = pd.read_csv(
    "../data/processed/nifty50_close_prices.csv",
    index_col="Date",
    parse_dates=True
)

## Select Cointegrated Pair for Backtesting


In [ ]:
# Example pair chosen from cointegration results
stock1 = close_prices["EICHERMOT.NS"]
stock2 = close_prices["MARUTI.NS"]


## Estimate Hedge Ratio (Beta)


In [ ]:
# Estimate hedge ratio using OLS regression
import statsmodels.api as sm

X = sm.add_constant(stock2)
model = sm.OLS(stock1, X).fit()

beta = model.params[stock2.name]
print("Beta:", beta)

## Construct Price Spread


In [ ]:
# Spread between two assets
spread = stock1 - beta * stock2


## Compute Z-Score of Spread


In [ ]:
# Rolling window used to normalize spread
window = 60


rolling_mean = spread.rolling(window).mean()
rolling_std = spread.rolling(window).std()

z_score = (spread - rolling_mean) / rolling_std

## Generate Trading Signals


In [ ]:
# Create signal dataframe
signals = pd.DataFrame(index=spread.index)

# Z-score signal
signals["zscore"] = z_score

# Long when spread very negative
signals["long"] = signals["zscore"] < -2

# Short when spread very positive
signals["short"] = signals["zscore"] > 2

signals=signals.dropna()
spread=spread.loc[signals.index]

## Build Trading Position


In [ ]:
signals["position"] = 0
signals.loc[signals["zscore"] < -2, "position"] = 1
signals.loc[signals["zscore"] > 2, "position"] = -1

## Compute Strategy Returns


In [ ]:
# Change in spread
spread_returns = spread.diff()

# Strategy returns
strategy_returns = signals["position"].shift(1) * spread_returns

# Cumulative performance
cumulative_returns = strategy_returns.cumsum()


## Strategy Equity Curve


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(cumulative_returns)

plt.title("Pairs Trading Strategy Returns")
plt.savefig("../reports/equity_curve.png")
plt.show()

## Strategy Performance Metrics


In [ ]:
total_return = cumulative_returns.iloc[-1]

sharpe_ratio = np.sqrt(252) * strategy_returns.mean() / strategy_returns.std()

drawdown = cumulative_returns / cumulative_returns.cummax() - 1
max_drawdown = drawdown.min()

win_rate = (strategy_returns > 0).sum() / len(strategy_returns)

print("Win Rate:", win_rate)


print("Total Return:", total_return)
print("Sharpe Ratio:", sharpe_ratio)
print("Max Drawdown:", max_drawdown)

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Total Return", "Sharpe Ratio", "Max Drawdown", "Win Rate"],
    "Value": [total_return, sharpe_ratio, max_drawdown, win_rate]
})

metrics.to_csv("../data/processed/strategy_metrics.csv", index=False)

print("Strategy metrics saved successfully")

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(drawdown)

plt.title("Portfolio Drawdown")

plt.savefig("../reports/portfolio_drawdown.png")

plt.show()

In [ ]:
results = pd.DataFrame({
    "spread": spread,
    "zscore": z_score,
    "position": signals["position"],
    "strategy_returns": strategy_returns,
    "cumulative_returns": cumulative_returns
})

results.to_csv("../data/processed/pair_performance.csv")

print("Strategy results saved successfully")


### Backtest Summary

This backtest evaluates the statistical arbitrage strategy between EICHERMOT and MARUTI.

Key Results:
- Total Return
- Sharpe Ratio
- Maximum Drawdown

These results will be used in later notebooks for portfolio analysis and advanced strategy evaluation.
